In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

from data.utils.fsdd import FSDD
from src.data.dataloader import dataUtils, FSDDDataset
from src.utils.config import Config
from src.data.feat_aug import Augmentation
from src.trainer.trainer import train_model
from src.model.mymodel import MLPMTLModel
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

In [2]:
# audio_dir="C:\\Users\\Ammar\\OneDrive\\Dokumen\\NextCloud\\My Documents\\Kuylah S2\\SEM 2\\RFPP\\RFPP_Task\\Last Project\\pseudo MTL\\data\\recordings\\"
# fsdd_tool = FSDD(audio_dir)

In [3]:
# spectr, label = fsdd_tool.get_spectrograms("C:\\Users\\Ammar\\OneDrive\\Dokumen\\NextCloud\\My Documents\\Kuylah S2\\SEM 2\\RFPP\\RFPP_Task\\Last Project\\pseudo MTL\\data\\spectrotest\\")
# print(len(spectr[0]))

In [4]:
DataUtils = dataUtils()
data = DataUtils.load_data(Config.AUDIO_PATH)

In [5]:
# extracted_feat = feature_extraction(data)

In [6]:
cols_to_fix = ['digit', 'sample_number'] 

data[cols_to_fix] = data[cols_to_fix].astype(int)

data = data.sort_values(
    by=['speaker', 'digit', 'sample_number'], 
    ascending=True
).reset_index(drop=True)

print(data.head())

                                                path speaker  digit  \
0  C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...  george      0   
1  C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...  george      0   
2  C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...  george      0   
3  C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...  george      0   
4  C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...  george      0   

   sample_number  
0              0  
1              1  
2              2  
3              3  
4              4  


In [7]:
train_data, test_data = DataUtils.data_split(data)

In [8]:
train_data.head()

,path,speaker,digit,sample_number
0,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,5
1,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,6
2,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,7
3,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,8
4,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,9


In [9]:
le = LabelEncoder()
train_data['speaker_id'] = le.fit_transform(train_data['speaker'])
test_data['speaker_id'] = le.transform(test_data['speaker'])

In [10]:
train_data.head()

,path,speaker,digit,sample_number,speaker_id
0,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,5,0
1,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,6,0
2,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,7,0
3,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,8,0
4,C:\Users\Ammar\OneDrive\Dokumen\NextCloud\My D...,george,0,9,0


In [11]:
aug = Augmentation()
    
# Bungkus fungsi agar hanya dieksekusi saat dipanggil di Dataset
audio_augmentations = [
    lambda y, sr: aug.add_gaussian_noise(y, noise_factor=0.005),
    lambda y, sr: aug.time_stretch(y, rate=1.1),
    lambda y, sr: aug.pitch_shift(y, sr, pitch_steps=[-1, 1])
]

mfcc_augmentations = [
    lambda m: aug.time_masking(m, max_mask_t=5),
    lambda m: aug.freq_masking(m, max_mask_f=2)
]

train_dataset = FSDDDataset(
    df=train_data, 
    audio_transform=audio_augmentations,
    mfcc_transform=mfcc_augmentations
)

test_dataset = FSDDDataset(
    df=test_data, 
    audio_transform=None,
    mfcc_transform=None
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
model = MLPMTLModel(input_size=40, num_classes_task1=10, num_classes_task2=6)
train_model(model=model, train_loader=train_loader, val_loader=None, epochs=10, lr=0.001, val_step=10)

Training started on device: cuda


c:\Users\Ammar\miniconda3\envs\rfpp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [1/10] - total_loss=8.4541 - l1=6.6409 - l2=4.5330
Epoch [2/10] - total_loss=3.6387 - l1=3.0004 - l2=1.5958
